# 🇹🇷 Türk İşaret Dili (TİD / AUTSL - 10 Sınıf) 5-Fold Çapraz Doğrulama & Ensemble Eğitimi
Bu notebook, **Signer-Independent 5-Fold Cross-Validation** mimarisini tek tuşla, sıfır hata ile çalıştırmak üzere optimize edilmiştir:
1. **Eğitim:** 25 işaretçi (1.068 video) `GroupKFold` ile 5 ayrı gruba ayrılır (Sıfır İşaretçi Sızıntısı).
2. **Kilitli Test Kasası:** 6 işaretçi (177 video) eğitime asla sokulmaz.
3. **Ensemble Testi:** 5 modelin olasılık tahminleri ortalanarak (Soft-Voting) nihai kilitli test başarımı ölçülür ve grafik çizdirilir.

### 1. Drive Bağlantısı ve Bağımlılıkların Kurulması

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q mediapipe opencv-python-headless rich tqdm PyYAML matplotlib scikit-learn

import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("✅ Drive bağlandı, bağımlılıklar yüklendi.")

### 2. Proje Kodlarının Colab Yerel Hızlı Diskine Kopyalanması

In [ ]:
import os, shutil

drive_root = "/content/drive/MyDrive/BitirmeProjesi"
local_root = "/content/BitirmeProjesi"

os.makedirs(local_root, exist_ok=True)

# 42 GB videolara dokunmadan sadece gereken kod klasörlerini kopyalar (1 saniye sürer)
for folder in ["src", "scripts", "configs"]:
    src_folder = os.path.join(drive_root, folder)
    dst_folder = os.path.join(local_root, folder)
    if os.path.exists(dst_folder):
        shutil.rmtree(dst_folder)
    shutil.copytree(src_folder, dst_folder)
    print(f"✅ {folder} klasörü kopyalandı.")

# Outputs klasörünü Drive'a bağla (modeller ve grafikler doğrudan Drive'a kaydedilir)
os.makedirs(f"{drive_root}/outputs", exist_ok=True)
if not os.path.exists(f"{local_root}/outputs"):
    os.symlink(f"{drive_root}/outputs", f"{local_root}/outputs")

print("✅ Outputs klasörü Drive'a bağlandı. Kodlar hazır!")

### 3. TİD (AUTSL) 10-Sınıf Video Paketini Açma
*(Daha önce Drive'a yüklediğimiz 411 MB'lık hazır `tid_10_sinif_paketi.zip` dosyasını `/content/tid_10_classes/` dizinine açar)*

In [ ]:
import os, zipfile

if not os.path.exists("/content/tid_10_classes/labels.csv"):
    zpath = "/content/drive/MyDrive/BitirmeProjesi/tid_10_sinif_paketi.zip"
    print(f"📦 Video paketi açılıyor: {zpath}")
    with zipfile.ZipFile(zpath, 'r') as zip_ref:
        zip_ref.extractall("/content/tid_10_classes")
    print("✅ 1.245 TİD videosu ve labels.csv başarıyla çıkarıldı!")
else:
    print("✅ /content/tid_10_classes zaten mevcut, açma adımı atlandı.")

### 4. Hazır Özellik Önbelleğini (Cache) Colab Hızlı Diskine Kopyalama
*(Drive'daki hazır 1.245 adet `.npz` dosyasını yerel SSD'ye aktarır — 5 saniye sürer)*

In [ ]:
# Drive'daki hazır cache dosyalarını Colab hızlı diskine kopyala
!mkdir -p /content/BitirmeProjesi/cache/tid_10_features
!cp -rn /content/drive/MyDrive/BitirmeProjesi/cache/tid_10_features/* /content/BitirmeProjesi/cache/tid_10_features/ 2>/dev/null || true

# Kontrol et (1245 dosya görmelisin):
!echo "Toplam Cache Dosyası: $(ls /content/BitirmeProjesi/cache/tid_10_features | wc -l)"

### 5. 🚀 5-Fold Çapraz Doğrulama & 5-Model Ensemble Eğitimi
- 5 kat sırayla eğitilir, her katın en iyi modeli (`best_model.pt`) kaydedilir.
- 5 model birleştirilerek (**Soft-Voting Ensemble**) kilitli 177 test videosunda nihai sınav yapılır.

In [ ]:
%cd /content/BitirmeProjesi

# 🚀 5-Fold Cross-Validation ve Ensemble Eğitimi Başlat
!python scripts/train_cv.py \
    --config configs/experiment/tid_10_mobilenet.yaml \
    --n_splits 5 \
    --cache_dir /content/BitirmeProjesi/cache/tid_10_features

### 6. Eğitim ve Çapraz Doğrulama Grafiğini Görüntüleme

In [ ]:
from IPython.display import Image, display
import os

cv_plot = "/content/drive/MyDrive/BitirmeProjesi/outputs/tid_10_mobilenet_cv/cv_results_plot.png"
if os.path.exists(cv_plot):
    print("📊 5-Fold Cross-Validation ve Ensemble Sonuç Grafiği:")
    display(Image(cv_plot))
else:
    print("Grafik dosyası henüz bulunamadı.")

### 7. 📦 Bağımsız Dağıtım Paketini Oluşturma (tid_autsl_10_sinif_paketi.zip)
Tüm kaynak kodları, konfigürasyonları ve 1.245 adet hazır MediaPipe (.npz) özellik önbelleğini tek bir zip dosyasında toplar.
Oluşan zip doğrudan Drive dizinine (/content/drive/MyDrive/BitirmeProjesi/tid_autsl_10_sinif_paketi.zip) kaydedilir.

In [ ]:
%cd /content/BitirmeProjesi
!python scripts/export_package.py